<a href="https://colab.research.google.com/github/david-levin11/Verification_Notebooks/blob/main/nbm_grib_download.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title ⛳ Optional: Mount Google Drive
MOUNT_DRIVE = False  #@param {type:"boolean"}

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted at /content/drive")
else:
    print("Skipping Drive mount. Files will be saved in /content (ephemeral).")


In [ ]:
#@title ⚙️ NBM f006 APCP subset downloader (functions)--Run just once
import os, re, sys
import datetime as dt
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests

BASE = "https://noaa-nbm-grib2-pds.s3.amazonaws.com"

def date_iter(year: int):
    d = dt.date(year, 1, 1)
    end = dt.date(year, 12, 31)
    one = dt.timedelta(days=1)
    while d <= end:
        yield d
        d += one

def build_url(date_obj: dt.date, hour: int, product: str = "core", region: str = "ak") -> str:
    ymd = date_obj.strftime("%Y%m%d")
    hh = f"{int(hour):02d}"
    fname = f"blend.t{hh}z.{product}.f006.{region}.grib2"
    return f"{BASE}/blend.{ymd}/{hh}/{product}/{fname}"

def ensure_parent_dir(path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)

def download_subset(remote_url: str,
                    search_strings,
                    local_filename: str,
                    exclude_phrase: str = None,
                    require_all_matches: bool = True,
                    timeout: int = 30) -> str | None:
    """Subset a GRIB2 using byte ranges from its .idx file."""
    remote_file = os.path.basename(remote_url)
    idx_url = remote_url + ".idx"

    try:
        r = requests.get(idx_url, timeout=timeout, headers={"User-Agent": "nbm-subsetter/1.0"})
    except Exception as e:
        print(f"❌ Failed to fetch idx {idx_url}: {e}")
        return None

    if not r.ok or not r.text.strip():
        print(f"⚠️ Missing or empty idx: {idx_url}")
        return None

    idx_lines = r.text.strip().splitlines()
    exprs = {s: re.compile(s) for s in search_strings}
    matched_ranges = {}
    matched_vars = set()

    for n, line in enumerate(idx_lines):
        if exclude_phrase and exclude_phrase in line:
            continue
        for s, rx in exprs.items():
            if rx.search(line):
                matched_vars.add(s)
                parts = line.split(':')
                try:
                    start = int(parts[1])
                except Exception:
                    continue
                if n + 1 < len(idx_lines):
                    parts_next = idx_lines[n + 1].split(':')
                    try:
                        end = int(parts_next[1]) - 1
                        b_range = f"{start}-{end}"
                    except Exception:
                        b_range = f"{start}-"
                else:
                    b_range = f"{start}-"
                matched_ranges[b_range] = line

    if require_all_matches and len(matched_vars) != len(search_strings):
        print(f"⚠️ Not all variables matched in {remote_file}. Found: {matched_vars}")
        return None
    if not matched_ranges:
        print(f"❌ No byte ranges matched in {remote_file}")
        return None

    ensure_parent_dir(local_filename)
    with open(local_filename, "wb") as f_out:
        for b_range in matched_ranges.keys():
            headers = {"Range": f"bytes={b_range}", "User-Agent": "nbm-subsetter/1.0"}
            try:
                rr = requests.get(remote_url, headers=headers, timeout=timeout)
            except Exception as e:
                print(f"❌ Range {b_range} failed for {remote_file}: {e}")
                return None
            if rr.status_code not in (200, 206):
                print(f"❌ HTTP {rr.status_code} on range {b_range} for {remote_file}")
                return None
            f_out.write(rr.content)

    if os.path.getsize(local_filename) > 10_000:
        print(f"✅ Downloaded [{len(matched_ranges)}] field(s) → {local_filename}")
        return local_filename
    else:
        print(f"❌ File too small or failed: {local_filename}")
        return None

def task(date_obj: dt.date, hour: int, out_dir: str, product: str, region: str):
    remote_url = build_url(date_obj, hour, product=product, region=region)
    ymd = date_obj.strftime("%Y%m%d")
    hh = f"{int(hour):02d}"
    local_rel = f"{ymd}/t{hh}z/blend.t{hh}z.{product}.f006.{region}.grib2"
    local_path = os.path.join(out_dir, local_rel)

    if os.path.exists(local_path) and os.path.getsize(local_path) > 10_000:
        print(f"⏩ Exists (skipping): {local_rel}")
        return remote_url, local_path

    search = [r":APCP:surface:0-6 hour acc fcst:"]
    result = download_subset(
        remote_url,
        search_strings=search,
        local_filename=local_path,
        exclude_phrase="prob",
        require_all_matches=True
    )
    return remote_url, result

def run_year(year: int,
             out_root: str,
             cycles: list[int] | str = "all",
             product: str = "core",
             region: str = "ak",
             workers: int = 8):
    if isinstance(cycles, str) and cycles.lower() == "all":
        cycles = list(range(24))
    else:
        cycles = [int(x) for x in cycles]

    jobs = []
    for d in date_iter(year):
        for h in cycles:
            jobs.append((d, h, out_root, product, region))

    total = len(jobs)
    print(f"Planned jobs: {total} (year={year}, cycles={cycles}, out='{out_root}')")

    ok = 0
    missing = 0
    with ThreadPoolExecutor(max_workers=int(workers)) as ex:
        futures = [ex.submit(task, *j) for j in jobs]
        for fut in as_completed(futures):
            remote_url, path = fut.result()
            if path is None:
                missing += 1
            else:
                ok += 1

    print("\n=== Summary ===")
    print(f"Successful subsets: {ok}")
    print(f"Skipped/failed:     {missing}")
    print(f"Output root:        {os.path.abspath(out_root)}")


In [ ]:
#@title 🚀 Run the downloader
YEAR = 2024            #@param {type:"integer"}
CYCLES = "00,06,12,18"         #@param ["all", "00,06,12,18", "00", "06", "12", "18"]
WORKERS = 8            #@param {type:"integer"}
PRODUCT = "core"       #@param {type:"string"}
REGION = "ak"          #@param {type:"string"}

# Choose an output directory
USE_DRIVE = MOUNT_DRIVE if 'MOUNT_DRIVE' in globals() else False
if USE_DRIVE:
    OUT_DIR = f"/content/drive/MyDrive/NBM_f006_APCP_{YEAR}"
else:
    OUT_DIR = f"/content/NBM_f006_APCP_{YEAR}"

# Parse cycles value
if CYCLES == "all":
    cycles_arg = "all"
elif "," in CYCLES:
    cycles_arg = CYCLES.split(",")
else:
    cycles_arg = [CYCLES]

print(f"Saving to: {OUT_DIR}")
run_year(YEAR, OUT_DIR, cycles=cycles_arg, product=PRODUCT, region=REGION, workers=WORKERS)


In [ ]:
#@title 📦 (Optional) Tar the output folder
ARCHIVE_NAME = "f\"NBM_PrecipData_2024.tar.gz\""  #@param {type:"string"}
!tar -czvf "{ARCHIVE_NAME}" -C "$(dirname '{OUT_DIR}')" "$(basename '{OUT_DIR}')"
print("Done. You can right-click the tarball in the Files panel to download.")
